# Ollama Worker Demo

This notebook demonstrates the structured local worker flow:

1. Create a task packet
2. Create a worker
3. Run the worker with a live Ollama-backed task
4. Inspect worker state
5. Resume the last packet
6. Close the worker

The notebook captures the `worker_id` programmatically. No manual copy/paste is required.

In [1]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
import urllib.error
import urllib.request
from pathlib import Path

def is_repo_root(candidate: Path) -> bool:
    return (candidate / "src" / "main.py").exists() and (candidate / "tests" / "test_porting_workspace.py").exists()

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if is_repo_root(candidate):
            return candidate
        for nested in (candidate / "claw-code", candidate / "harness" / "claw-code"):
            if is_repo_root(nested):
                return nested
    raise RuntimeError(
        f"Could not locate the claw-code repo from {current}. Start the notebook from the repo, its notebooks directory, or the enclosing autoresearch workspace."
    )

ROOT = find_repo_root()
PYTHON = sys.executable
NOTEBOOK_DIR = ROOT / "notebooks"
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

def run_cmd(args: list[str], check: bool = True) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(shlex.quote(arg) for arg in args))
    result = subprocess.run(args, cwd=ROOT, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        details = result.stderr.strip() or result.stdout.strip() or "no output"
        raise RuntimeError(f"command failed with exit code {result.returncode}: {details}")
    return result

def run_json(args: list[str], check: bool = True):
    result = run_cmd(args, check=check)
    return json.loads(result.stdout)

def ollama_ready(host: str = "http://localhost:11434") -> bool:
    try:
        with urllib.request.urlopen(f"{host.rstrip('/')}/api/tags", timeout=2) as response:
            return response.status == 200
    except (urllib.error.URLError, TimeoutError):
        return False

print(f"Workspace root: {ROOT}")
print(f"Python: {PYTHON}")

Workspace root: /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code
Python: /Users/wongdowling/opt/anaconda3/bin/python


In [2]:
OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5-coder:7b"
OLLAMA_IS_READY = ollama_ready(OLLAMA_HOST)
print({"ollama_host": OLLAMA_HOST, "ollama_ready": OLLAMA_IS_READY})
if not OLLAMA_IS_READY:
    raise RuntimeError("Ollama is not reachable. Start Ollama before running the live smoke test.")

{'ollama_host': 'http://localhost:11434', 'ollama_ready': True}


In [3]:
task_packet = {
    "objective": "Inspect the Python CLI and report the worker-related commands.",
    "scope": "Inspect src/main.py and worker modules only when needed.",
    "repo": "claw-code",
    "branch_policy": "Do not create or switch branches.",
    "acceptance_tests": [
        "python3 -m unittest tests/test_porting_workspace.py"
    ],
    "commit_policy": "Do not commit.",
    "reporting_contract": "Report worker commands, changed files, verification results, and blockers.",
    "escalation_policy": "Stop if Ollama is unavailable or a required tool fails repeatedly."
}

task_path = NOTEBOOK_DIR / "demo_task.json"
task_path.write_text(json.dumps(task_packet, indent=2), encoding="utf-8")
task_path

PosixPath('/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/notebooks/demo_task.json')

In [4]:
worker = run_json([
    PYTHON,
    "-m",
    "src.main",
    "worker",
    "create",
    "--root",
    str(ROOT),
    "--model",
    OLLAMA_MODEL,
    "--host",
    OLLAMA_HOST,
])
worker_id = worker["worker_id"]
worker

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main worker create --root /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code --model qwen2.5-coder:7b --host http://localhost:11434
{
  "worker_id": "07786cd4905d4275ab8666a113bf7e10",
  "root": "/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code",
  "model": "qwen2.5-coder:7b",
  "host": "http://localhost:11434",
  "state": "ready",
  "created_at": "2026-04-07T20:09:34Z",
  "updated_at": "2026-04-07T20:09:34Z",
  "run_count": 0,
  "last_packet": null,
  "last_result": null,
  "last_error": null
}



{'worker_id': '07786cd4905d4275ab8666a113bf7e10',
 'root': '/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code',
 'model': 'qwen2.5-coder:7b',
 'host': 'http://localhost:11434',
 'state': 'ready',
 'created_at': '2026-04-07T20:09:34Z',
 'updated_at': '2026-04-07T20:09:34Z',
 'run_count': 0,
 'last_packet': None,
 'last_result': None,
 'last_error': None}

In [5]:
worker_run = run_json([
    PYTHON,
    "-m",
    "src.main",
    "worker",
    "run",
    worker_id,
    "--packet",
    str(task_path),
    "--trace",
])
worker_run["state"], worker_run["last_result"]["stop_reason"]

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main worker run 07786cd4905d4275ab8666a113bf7e10 --packet /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/notebooks/demo_task.json --trace
{
  "worker_id": "07786cd4905d4275ab8666a113bf7e10",
  "root": "/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code",
  "model": "qwen2.5-coder:7b",
  "host": "http://localhost:11434",
  "state": "finished",
  "created_at": "2026-04-07T20:09:34Z",
  "updated_at": "2026-04-07T20:10:20Z",
  "run_count": 1,
  "last_packet": {
    "objective": "Inspect the Python CLI and report the worker-related commands.",
    "scope": "Inspect src/main.py and worker modules only when needed.",
    "repo": "claw-code",
    "branch_policy": "Do not create or switch branches.",
    "acceptance_tests": [
      "python3 -m unittest tests/test_porting_workspace.py"
    ],
    "commit_policy": "Do not commit.",
    "reporting_contract": "Report worker commands, changed files, verificati

('finished', 'completed')

In [6]:
result = worker_run["last_result"]
print("tool_calls:", result["tool_calls"])
print("changed_files:", result["changed_files"])
print("verification:")
print(json.dumps(result["verification"], indent=2))
print("final_answer:")
print(result["final_answer"])


tool_calls: ['read_file']
changed_files: []
verification:
{
  "acceptance_tests": [
    {
      "command": "python3 -m unittest tests/test_porting_workspace.py",
      "success": true,
      "exit_code": 0,
      "output": ".........................................\n----------------------------------------------------------------------\nRan 41 tests in 4.739s\n\nOK"
    }
  ],
  "tool_runs": {
    "run_tests": [],
    "run_build": []
  }
}
final_answer:
I found a Python script at 'src/main.py' that serves as the main entry point for a CLI application. It includes various subcommands and functionalities such as building parsers, resolving chat prompts, and executing commands like 'summary', 'manifest', 'parity-audit', etc.


In [7]:
worker_status = run_json([
    PYTHON,
    "-m",
    "src.main",
    "worker",
    "status",
    worker_id,
])
worker_status["state"]

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main worker status 07786cd4905d4275ab8666a113bf7e10
{
  "worker_id": "07786cd4905d4275ab8666a113bf7e10",
  "root": "/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code",
  "model": "qwen2.5-coder:7b",
  "host": "http://localhost:11434",
  "state": "finished",
  "created_at": "2026-04-07T20:09:34Z",
  "updated_at": "2026-04-07T20:10:20Z",
  "run_count": 1,
  "last_packet": {
    "objective": "Inspect the Python CLI and report the worker-related commands.",
    "scope": "Inspect src/main.py and worker modules only when needed.",
    "repo": "claw-code",
    "branch_policy": "Do not create or switch branches.",
    "acceptance_tests": [
      "python3 -m unittest tests/test_porting_workspace.py"
    ],
    "commit_policy": "Do not commit.",
    "reporting_contract": "Report worker commands, changed files, verification results, and blockers.",
    "escalation_policy": "Stop if Ollama is unavailable or a required tool fail

'finished'

In [8]:
worker_resumed = run_json([
    PYTHON,
    "-m",
    "src.main",
    "worker",
    "resume",
    worker_id,
    "--trace",
])
worker_resumed["run_count"]

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main worker resume 07786cd4905d4275ab8666a113bf7e10 --trace
{
  "worker_id": "07786cd4905d4275ab8666a113bf7e10",
  "root": "/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code",
  "model": "qwen2.5-coder:7b",
  "host": "http://localhost:11434",
  "state": "finished",
  "created_at": "2026-04-07T20:09:34Z",
  "updated_at": "2026-04-07T20:10:59Z",
  "run_count": 2,
  "last_packet": {
    "objective": "Inspect the Python CLI and report the worker-related commands.",
    "scope": "Inspect src/main.py and worker modules only when needed.",
    "repo": "claw-code",
    "branch_policy": "Do not create or switch branches.",
    "acceptance_tests": [
      "python3 -m unittest tests/test_porting_workspace.py"
    ],
    "commit_policy": "Do not commit.",
    "reporting_contract": "Report worker commands, changed files, verification results, and blockers.",
    "escalation_policy": "Stop if Ollama is unavailable or a required t

2

In [9]:
worker_closed = run_json([
    PYTHON,
    "-m",
    "src.main",
    "worker",
    "close",
    worker_id,
])
worker_closed["state"]

$ /Users/wongdowling/opt/anaconda3/bin/python -m src.main worker close 07786cd4905d4275ab8666a113bf7e10
{
  "worker_id": "07786cd4905d4275ab8666a113bf7e10",
  "root": "/Users/wongdowling/Documents/autoresearch_harness/harness/claw-code",
  "model": "qwen2.5-coder:7b",
  "host": "http://localhost:11434",
  "state": "closed",
  "created_at": "2026-04-07T20:09:34Z",
  "updated_at": "2026-04-07T20:10:59Z",
  "run_count": 2,
  "last_packet": {
    "objective": "Inspect the Python CLI and report the worker-related commands.",
    "scope": "Inspect src/main.py and worker modules only when needed.",
    "repo": "claw-code",
    "branch_policy": "Do not create or switch branches.",
    "acceptance_tests": [
      "python3 -m unittest tests/test_porting_workspace.py"
    ],
    "commit_policy": "Do not commit.",
    "reporting_contract": "Report worker commands, changed files, verification results, and blockers.",
    "escalation_policy": "Stop if Ollama is unavailable or a required tool fails r

'closed'

## Smoke Test Success Criteria

The smoke test is considered successful when:

- the worker is created without manual `worker_id` handling
- `worker run` returns JSON
- `last_result.tool_calls` is non-empty
- `last_result.verification.acceptance_tests` is present
- `last_result.stop_reason` is `completed` or `verification_failed`
- `worker status`, `worker resume`, and `worker close` all succeed